In [ ]:
# Install library untuk perceptual hashing (jika belum ada)
!pip install -q imagehash Pillow

import os
import hashlib
import imagehash
from PIL import Image
from collections import defaultdict
import shutil

In [ ]:
def find_duplicates_in_folder(folder_path, similarity_threshold=5):
    """
    Mencari gambar duplikat atau sangat mirip di dalam folder.
    Return: (list_of_duplicate_groups, list_of_files_to_remove)
    """
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return [], []  # <-- Kembalikan dua list kosong

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    file_paths = [os.path.join(folder_path, f) for f in files]

    # Step 1: Exact duplicate
    hash_dict = defaultdict(list)
    for fp in file_paths:
        with open(fp, 'rb') as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        hash_dict[md5].append(fp)

    exact_duplicates = [group for group in hash_dict.values() if len(group) > 1]

    files_to_remove = set()
    for group in exact_duplicates:
        group_sorted = sorted(group)
        for dup in group_sorted[1:]:
            files_to_remove.add(dup)

    # Step 2: Near-duplicate
    remaining_files = [fp for fp in file_paths if fp not in files_to_remove]
    hashes = {}
    for fp in remaining_files:
        try:
            img = Image.open(fp)
            hashes[fp] = imagehash.phash(img)
        except Exception:
            continue

    near_duplicate_groups = []
    processed = set()
    fps = list(hashes.keys())
    for i in range(len(fps)):
        if fps[i] in processed:
            continue
        group = [fps[i]]
        for j in range(i+1, len(fps)):
            if fps[j] in processed:
                continue
            dist = hashes[fps[i]] - hashes[fps[j]]
            if dist <= similarity_threshold:
                group.append(fps[j])
                processed.add(fps[j])
        if len(group) > 1:
            near_duplicate_groups.append(group)
            processed.add(fps[i])

    for group in near_duplicate_groups:
        group_sorted = sorted(group)
        for dup in group_sorted[1:]:
            files_to_remove.add(dup)

    all_duplicate_groups = exact_duplicates + near_duplicate_groups
    return all_duplicate_groups, list(files_to_remove)

In [ ]:
def find_cross_duplicates(folder1, folder2, similarity_threshold=5):
    """
    Mencari gambar yang mirip antara folder1 dan folder2 menggunakan perceptual hash.
    Return: list of tuples (path1, path2, distance)
    """
    if not os.path.exists(folder1) or not os.path.exists(folder2):
        print("Folder tidak ditemukan.")
        return []

    files1 = [f for f in os.listdir(folder1) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    files2 = [f for f in os.listdir(folder2) if f.lower().endswith(('.jpg','.jpeg','.png'))]

    hashes1 = {}
    for f in files1:
        fp = os.path.join(folder1, f)
        try:
            img = Image.open(fp)
            hashes1[fp] = imagehash.phash(img)
        except Exception as e:
            pass

    hashes2 = {}
    for f in files2:
        fp = os.path.join(folder2, f)
        try:
            img = Image.open(fp)
            hashes2[fp] = imagehash.phash(img)
        except Exception as e:
            pass

    cross_matches = []
    for fp1, h1 in hashes1.items():
        for fp2, h2 in hashes2.items():
            dist = h1 - h2
            if dist <= similarity_threshold:
                cross_matches.append((fp1, fp2, dist))

    return cross_matches

In [ ]:
def detect_multiple_objects(image_path, min_area_ratio=0.1):
    """
    Mendeteksi apakah ada lebih dari satu kemasan obat dalam gambar.
    Return: (jumlah_objek, list_bounding_box)
    """
    img = cv2.imread(image_path)
    if img is None:
        return 0, []

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(edges, kernel, iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return 0, []

    h, w = img.shape[:2]
    min_area = (h * w) * min_area_ratio

    valid_objects = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        peri = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
        if len(approx) >= 4:  # Bentuk persegi atau lebih
            x, y, w_rect, h_rect = cv2.boundingRect(approx)
            valid_objects.append((x, y, w_rect, h_rect))

    return len(valid_objects), valid_objects

In [ ]:
def detect_incomplete_package(image_path):
    """
    Mendeteksi apakah kemasan terpotong/tdk utuh.
    Ciri: kontur besar menyentuh tepi gambar.
    Return: (is_incomplete, bounding_box)
    """
    img = cv2.imread(image_path)
    if img is None:
        return False, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(edges, kernel, iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return False, None

    h, w = img.shape[:2]
    margin = 15  # piksel dari tepi

    # Cari kontur terbesar
    max_area = 0
    best_rect = None
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > max_area:
            max_area = area
            x, y, w_rect, h_rect = cv2.boundingRect(contour)
            best_rect = (x, y, w_rect, h_rect)

    if best_rect is None:
        return False, None

    x, y, w_rect, h_rect = best_rect

    # Cek apakah bounding box menyentuh tepi
    if (x <= margin or y <= margin or
        (x + w_rect) >= (w - margin) or
        (y + h_rect) >= (h - margin)):
        return True, best_rect

    return False, best_rect

In [ ]:
def detect_extreme_lighting(image_path, brightness_low=30, brightness_high=225):
    """
    Mendeteksi gambar dengan pencahayaan ekstrim (terlalu gelap atau backlight).
    Return: (is_extreme, category, brightness_mean)
    """
    img = cv2.imread(image_path)
    if img is None:
        return False, "error", 0

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    brightness_mean = np.mean(gray)

    if brightness_mean < brightness_low:
        return True, "terlalu_gelap", brightness_mean
    elif brightness_mean > brightness_high:
        return True, "terlalu_terang", brightness_mean
    else:
        return False, "normal", brightness_mean

In [ ]:
base_dir = '/content/drive/MyDrive/BlockchainAI/dataset'
asli_dir = os.path.join(base_dir, 'asli')
palsu_dir = os.path.join(base_dir, 'palsu')

print("="*60)
print("MEMERIKSA DUPLIKAT DI FOLDER ASLI")
print("="*60)
dup_groups_asli, files_to_remove_asli = find_duplicates_in_folder(asli_dir, similarity_threshold=5)

if dup_groups_asli:
    print(f"Ditemukan {len(dup_groups_asli)} grup duplikat.")
    for i, group in enumerate(dup_groups_asli):
        print(f"Grup {i+1}: {len(group)} file")
        for fp in group:
            print(f"  - {fp}")
else:
    print("Tidak ada duplikat ditemukan.")

# Hapus file duplikat
if files_to_remove_asli:
    print(f"\nMenghapus {len(files_to_remove_asli)} file duplikat dari folder asli...")
    for fp in files_to_remove_asli:
        os.remove(fp)
        print(f"Dihapus: {fp}")
else:
    print("Tidak ada file yang dihapus.")

In [ ]:
print("\n" + "="*60)
print("MEMERIKSA DUPLIKAT DI FOLDER PALSU")
print("="*60)
dup_groups_palsu, files_to_remove_palsu = find_duplicates_in_folder(palsu_dir, similarity_threshold=5)

if dup_groups_palsu:
    print(f"Ditemukan {len(dup_groups_palsu)} grup duplikat.")
    for i, group in enumerate(dup_groups_palsu):
        print(f"Grup {i+1}: {len(group)} file")
        for fp in group:
            print(f"  - {fp}")
else:
    print("Tidak ada duplikat ditemukan.")

if files_to_remove_palsu:
    print(f"\nMenghapus {len(files_to_remove_palsu)} file duplikat dari folder palsu...")
    for fp in files_to_remove_palsu:
        os.remove(fp)
        print(f"Dihapus: {fp}")
else:
    print("Tidak ada file yang dihapus.")

In [ ]:
print("\n" + "="*60)
print("MEMERIKSA GAMBAR BERMASALAH (MULTI-OBJEK, TIDAK UTUH, EKSTRIM)")
print("="*60)

quarantine_dir = os.path.join(base_dir, 'quarantine')
os.makedirs(quarantine_dir, exist_ok=True)

problems_log = []

for folder in ['asli', 'palsu']:
    folder_path = os.path.join(base_dir, folder)
    if not os.path.exists(folder_path):
        continue

    print(f"\nMemeriksa folder: {folder}")
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]

    for fname in files:
        filepath = os.path.join(folder_path, fname)
        reasons = []

        # 1. Cek multi-objek
        obj_count, _ = detect_multiple_objects(filepath)
        if obj_count > 1:
            reasons.append(f"Multi-objek ({obj_count} kemasan)")

        # 2. Cek tidak utuh
        is_incomplete, _ = detect_incomplete_package(filepath)
        if is_incomplete:
            reasons.append("Kemasan tidak utuh (terpotong)")

        # 3. Cek pencahayaan ekstrim
        is_extreme, category, brightness = detect_extreme_lighting(filepath)
        if is_extreme:
            reasons.append(f"Pencahayaan ekstrim ({category}, brightness={brightness:.1f})")

        # Jika ada masalah, pindahkan ke quarantine
        if reasons:
            dst = os.path.join(quarantine_dir, f"{folder}_{fname}")
            shutil.move(filepath, dst)
            problems_log.append((folder, fname, reasons))
            print(f"🚫 [{folder}] {fname} -> {'; '.join(reasons)}")

print("\n" + "="*60)
print(f"RINGKASAN: {len(problems_log)} gambar bermasalah dipindahkan ke {quarantine_dir}")

In [ ]:
print("\n" + "="*60)
print("MEMBERSIHKAN NAMA FILE")
print("="*60)
import re

def clean_filenames(folder_path):
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return
    count = 0
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            nama_baru = re.sub(r'[^a-zA-Z0-9_.]', '_', fname)
            nama_baru = re.sub(r'_+', '_', nama_baru)
            if nama_baru != fname:
                src = os.path.join(folder_path, fname)
                dst = os.path.join(folder_path, nama_baru)
                os.rename(src, dst)
                count += 1
                print(f"Renamed: {fname} → {nama_baru}")
    print(f"Total file di-rename: {count}")

for folder in ['asli', 'palsu']:
    print(f"\nMembersihkan folder: {folder}")
    clean_filenames(os.path.join(base_dir, folder))

demo_dir = '/content/drive/MyDrive/BlockchainAI/demo_test'
if os.path.exists(demo_dir):
    print(f"\nMembersihkan folder: demo_test")
    clean_filenames(demo_dir)

print("\n✅ Semua nama file sudah dibersihkan.")